# Mar 10/11 · 14:00 — TP v0.8 : ping par subprocess et statut des services

Mar 10/11, TP de 14:30 à 15:25. Correction à 15:25, TP fini ou non. La référence v0.8 est publiée à 15:30.

## Objectif

`fleetcheck` contrôle enfin quelque chose : chaque serveur est pingé.

- `ping` passe par `subprocess.run`, avec un délai maximal et l'option propre au système ;
- chaque service reçoit un statut (`ok` ou `down`) et ses raisons, le parc un statut global ;
- le code de sortie devient `1` quand un service est `down`, et la sortie texte donne une ligne par service.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook.
- `run(...)` lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`. Il utilise lui-même `subprocess.run` : regardez la cellule d'outils.
- Une exécution complète de `fleetcheck` dure environ une seconde : le délai du ping vers `db-1` (`192.0.2.1`), qui ne répond jamais.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push des mêmes fichiers qu'en v0.7 et de ce notebook, sans `reports/`.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` de v0.7, avec `config.json`, `requirements.txt`, `.gitignore` et les deux inventaires.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.7` du dépôt public.

Dans `servers.csv`, huit serveurs sont sur `127.0.0.1`, toujours joignable, et `db-1` sur `192.0.2.1`, une adresse réservée à la documentation, jamais routée : le résultat ne dépend pas du réseau de l'école.

Vérifiez avant de commencer :

In [ ]:
import json
from pathlib import Path

%run -n fleetcheck.py
check((callable(write_report), callable(archive_old), callable(exit_code)), (True, True, True))
config = json.loads(Path("config.json").read_text(encoding="utf-8"))
check(config["ping_timeout_s"], 1)

## Étapes

| Fin | Étape |
|---|---|
| 14:50 | 1. `ping(ip, timeout_s)` par `subprocess.run` |
| 15:05 | 2. `check_service(server, config)` et `global_status(statuses)` |
| 15:25 | 3. `main()` contrôle chaque serveur, écrit le rapport enrichi et sort avec le bon code |

À 14:55, si plus d'un tiers de la salle n'a pas un `ping` qui renvoie `False` sur `192.0.2.1` sans bloquer, l'enseignant écrit le `try`/`except` au tableau ; `as_text` est fourni (étape 3).

### 1. `ping` — fin 14:50

`ping(ip, timeout_s)` renvoie `True` si l'hôte répond, `False` sinon :

- commande en liste : `["ping", "-n", "1", ip]` sous Windows, `["ping", "-c", "1", ip]` ailleurs ; `platform.system()` renvoie `"Windows"`, `"Darwin"` ou `"Linux"` ;
- `subprocess.run(..., capture_output=True, timeout=timeout_s)` : la sortie de `ping` ne s'affiche pas, et le processus est tué au-delà du délai ;
- `True` si `returncode` vaut `0` ; `False` si le code est non nul, ou si `subprocess.TimeoutExpired` est levée.

Jamais `shell=True` : l'IP vient d'un fichier, et un shell interpréterait ce qu'elle contient.

**Prédire** — Sans `try`/`except`, `ping("192.0.2.1", 1)` renvoie-t-il `False` ? Sinon, que se passe-t-il quand `fleetcheck` tourne ?

In [ ]:
# Ma réponse :

**Justifier** — Pourquoi `["ping", "-c", "1", ip]` plutôt que `subprocess.run(f"ping -c 1 {ip}", shell=True)` ?

In [ ]:
# Ma réponse :

Vérification : `127.0.0.1` répond, `192.0.2.1` non, et la réponse arrive en moins de 2 s.

In [ ]:
import time

%run -n fleetcheck.py
check(ping("127.0.0.1", 1), True)
debut = time.monotonic()
resultat = ping("192.0.2.1", 1)
check((resultat, time.monotonic() - debut < 2), (False, True))

### 2. `check_service` et `global_status` — fin 15:05

`check_service(server, config)` renvoie une copie du serveur enrichie de trois clés, sans modifier le dict reçu :

| Clé | Ping OK | Ping KO |
|---|---|---|
| `status` | `"ok"` | `"down"` |
| `reasons` | `[]` | `["ping KO"]` |
| `ping` | `{"ok": True}` | `{"ok": False}` |

`{**server, "status": "ok"}` copie `server` et y ajoute la clé. Le délai vient de `config["ping_timeout_s"]`.

`global_status(statuses)` renvoie le pire statut de la liste, `"ok"` si elle est vide. La gravité ne suit pas l'ordre alphabétique (`max(["ok", "down"])` vaut `"ok"`) : comparez des gravités.

```text
SEVERITY = {"ok": 0, "degraded": 1, "down": 2}

max(statuses, key=SEVERITY.__getitem__, default="ok")
```

`key=` compare `SEVERITY["ok"]`, `SEVERITY["down"]`… au lieu des chaînes ; `default=` évite le `ValueError` de `max` sur une liste vide. `degraded` arrive en v0.9 avec `GET /health` ; `global_status` le gère déjà.

Vérification :

In [ ]:
%run -n fleetcheck.py
db = {"name": "db-1", "env": "prod", "ip": "192.0.2.1", "port": 5432, "role": "db"}
web = {"name": "web-1", "env": "prod", "ip": "127.0.0.1", "port": 8001, "role": "web"}
check(check_service(db, config), {**db, "status": "down", "reasons": ["ping KO"], "ping": {"ok": False}})
check(check_service(web, config), {**web, "status": "ok", "reasons": [], "ping": {"ok": True}})
check("status" in db, False)
check((global_status(["ok", "down", "ok"]), global_status(["ok", "degraded"]), global_status([])),
      ("down", "degraded", "ok"))

### 3. `main()` : contrôle, rapport, code de sortie — fin 15:25

Dans `main()`, après le chargement et le filtre `--env` :

1. `check_service` sur chaque serveur valide ; log `INFO <nom> : ok` si le service est `ok`, sinon `WARNING <nom> : <statut> <raisons>` (par exemple `WARNING db-1 : down ping KO`) ;
2. statut global par `global_status`, code par `exit_code(statuses, len(invalid))` ;
3. rapport enrichi, écrit par `write_report` (`3-fil-rouge/spec-statut-rapport.md` du cours) :

```text
{
  "generated_at": "...",
  "inventory": "servers.csv",
  "env": null,
  "summary": {"status": "down", "exit_code": 1, "total": 9, "ok": 8, "degraded": 0, "down": 1, "invalid": 0},
  "services": [ ...les serveurs enrichis par check_service... ],
  "invalid_lines": [ ...comme avant... ]
}
```

4. `--format text` affiche `as_text(report)`, `--format json` le rapport ; puis `sys.exit(code)`. Le résumé de 4 lignes de v0.1 à v0.7 disparaît, avec `by_role`, `count_by`, `format_counts` et `print_summary` s'ils ne servent plus.

`as_text`, à copier tel quel dans `fleetcheck.py` : une ligne par service, une par ligne invalide, une ligne vide, puis le verdict.

```text
def as_text(report):
    lines = [f"{s['status']:<9} {s['name']:<12} {s['env']:<8} {', '.join(s['reasons'])}".rstrip()
             for s in report["services"]]
    lines += [f"invalide  ligne {i['line']} : {i['error']}" for i in report["invalid_lines"]]
    sm = report["summary"]
    lines.append(f"\n{sm['status'].upper()} : {sm['ok']} ok, {sm['degraded']} degraded, {sm['down']} down, "
                 f"{sm['invalid']} invalide(s)")
    return "\n".join(lines)
```

`f"{x:<9}"` aligne `x` à gauche sur 9 caractères ; `.rstrip()` retire les espaces de fin quand il n'y a pas de raison.

**Prédire** — `db-1` est `down`, et `servers-broken.csv` contient 3 lignes invalides. Quel code de sortie pour `run("--inventory", "servers-broken.csv")` ?

In [ ]:
# Ma réponse :

Vérification, inventaire complet : 8 services `ok`, `db-1` `down`, code `1`.

In [ ]:
SORTIE = """ok        web-1        prod
ok        web-2        prod
ok        api-1        prod
ok        api-2        prod
ok        worker-1     prod
ok        web-stg-1    staging
ok        api-stg-1    staging
ok        cache-1      prod
down      db-1         prod     ping KO

DOWN : 8 ok, 0 degraded, 1 down, 0 invalide(s)
"""

r = run()
check((r.stdout, r.stderr, r.returncode), (SORTIE, "WARNING db-1 : down ping KO\n", 1))

Vérification, `--env staging` : aucun service `down`, code `0`.

In [ ]:
r = run("--env", "staging")
check((r.stdout, r.stderr, r.returncode), ("""ok        web-stg-1    staging
ok        api-stg-1    staging

OK : 2 ok, 0 degraded, 0 down, 0 invalide(s)
""", "", 0))

Vérification, `servers-broken.csv` : les lignes invalides s'ajoutent avant le verdict.

In [ ]:
r = run("--inventory", "servers-broken.csv")
lignes = r.stdout.splitlines()
check(lignes[-1], "DOWN : 8 ok, 0 degraded, 1 down, 3 invalide(s)")
check(sum(l.startswith("invalide  ligne ") for l in lignes), 3)
check((r.stderr.count("WARNING ligne"), "WARNING db-1 : down ping KO" in r.stderr, r.returncode), (3, True, 2))

Vérification, `--format json` : le résumé du rapport.

In [ ]:
r = run("--format", "json")
check(json.loads(r.stdout)["summary"],
      {"status": "down", "exit_code": 1, "total": 9, "ok": 8, "degraded": 0, "down": 1, "invalid": 0})
check(r.returncode, 1)

## Critère final

| Lancement | stdout | stderr | Code |
|---|---|---|---|
| `run()` | 9 lignes de service, puis `DOWN : 8 ok, 0 degraded, 1 down, 0 invalide(s)` | `WARNING db-1 : down ping KO` | `1` |
| `run("--env", "staging")` | 2 lignes `ok`, puis `OK : 2 ok, 0 degraded, 0 down, 0 invalide(s)` | rien | `0` |
| `run("--inventory", "servers-broken.csv")` | 9 services, 3 `invalide`, verdict `DOWN … 3 invalide(s)` | 3 `WARNING` de lignes, 1 de service | `2` |

Ce sont les vérifications de l'étape 3.

## Pour aller plus loin

- Les neuf pings se font l'un après l'autre : avec cent serveurs injoignables, `fleetcheck` attendrait cent secondes. Mesurez le temps d'un run (`time.monotonic()` autour de `run()`), puis cherchez dans la documentation de `concurrent.futures` comment lancer les pings en parallèle.
- Un réseau qui filtre l'ICMP rend `down` un serveur sain. Quelle information manque pour trancher ? Réponse en v0.9, avec `GET /health`.